# Figure S14

Draws drought-response distributions and the multimetric response space.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.decomposition import PCA


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')


ROOT = find_repo_root()
METRICS_PATH = ROOT / 'outputs' / 'RECON_MAIN_2011_2023' / 'metrics' / 'drought_metrics.csv'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS14'
OUT_DIR.mkdir(parents=True, exist_ok=True)
DISTRIBUTIONS_PATH = OUT_DIR / 'FigS14_distributions.png'
RESPONSE_SPACE_PATH = OUT_DIR / 'FigS14_response_space.png'

FEATURES = [
    'decline_m',
    'Rdown_m_per_month',
    'RR_early',
    'T50_months',
    'RR2019',
]
SHORT_LABELS = [
    'Maximum decline',
    'Decline rate',
    'Early recovery',
    'Half-recovery time',
    'Long-term recovery ratio',
]
COLORS = ['#C97845', '#A94F3D', '#258A87', '#76558A', '#3D7F55']

SPACE_CMAP = LinearSegmentedColormap.from_list(
    'figs8_density', ['#F5F7F7', '#B8CED4', '#5F8FA0', '#274F67'], N=256,
)

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 8.0,
    'axes.labelsize': 8.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'axes.linewidth': 0.7,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size': 2.8,
    'ytick.major.size': 2.8,
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})


In [ ]:
metrics = pd.read_csv(METRICS_PATH)
missing = [column for column in FEATURES if column not in metrics.columns]
if missing:
    raise KeyError(f'Missing drought metrics: {missing}')

complete = metrics[FEATURES].replace([np.inf, -np.inf], np.nan).dropna().copy()
if complete.empty:
    raise ValueError('No grid cells have all five drought-response metrics.')

rank_input = complete.copy()
rank_input['RR_early'] = rank_input['RR_early'].clip(0.0, 2.0)
rank_input['T50_months'] = rank_input['T50_months'].clip(1.0, 72.0)
rank_input['RR2019'] = rank_input['RR2019'].clip(-1.0, 5.0)
rank_values = rank_input.rank(pct=True, method='average')
X = (rank_values - rank_values.mean()) / rank_values.std(ddof=0)

pca = PCA(n_components=2)
scores = pca.fit_transform(X.to_numpy(dtype=float))
loadings = pca.components_.T.copy()

if loadings[0, 0] < 0:
    scores[:, 0] *= -1
    loadings[:, 0] *= -1
if loadings[1, 1] < 0:
    scores[:, 1] *= -1
    loadings[:, 1] *= -1

print(f'Complete five-metric grid cells: {len(complete):,} ({len(complete) / len(metrics) * 100:.2f}%)')
print('PCA explained variance:', ', '.join(f'PC{i + 1}={value:.3f}' for i, value in enumerate(pca.explained_variance_ratio_)))
print('No class labels or class boundaries are used in this figure.')


In [ ]:
distribution_specs = [
    dict(column='decline_m', xlabel='Maximum decline (m)', limits=(0.3, 5.0), bins=38, median_fmt='.2f'),
    dict(column='Rdown_m_per_month', xlabel='Decline rate (m / month)', limits=(0.04, 1.0), bins=38, median_fmt='.2f'),
    dict(column='RR_early', xlabel='Early recovery ratio', limits=(0.0, 2.0), bins=38, median_fmt='.2f'),
    dict(column='T50_months', xlabel='Half-recovery time (months)', limits=(1.0, 72.0), bins=np.geomspace(1.0, 72.0, 27), median_fmt='.0f', log_x=True),
    dict(column='RR2019', xlabel='Long-term recovery ratio', limits=(-1.0, 5.0), bins=40, median_fmt='.2f'),
]

fig_dist = plt.figure(figsize=(11.6, 2.75), facecolor='white')
dist_grid = fig_dist.add_gridspec(1, 5, left=0.060, right=0.985, bottom=0.235, top=0.955, wspace=0.34)

hist_axes = []
for index, (spec, color) in enumerate(zip(distribution_specs, COLORS)):
    ax = fig_dist.add_subplot(dist_grid[0, index])
    hist_axes.append(ax)
    raw = complete[spec['column']].to_numpy(dtype=float)
    lower, upper = spec['limits']
    displayed = np.clip(raw, lower, upper)
    weights = np.full(displayed.shape, 100.0 / displayed.size, dtype=float)
    counts, _, _ = ax.hist(
        displayed, bins=spec['bins'], weights=weights,
        color=color, alpha=0.84, edgecolor='white', linewidth=0.28,
    )
    median = float(np.nanmedian(raw))
    ax.axvline(np.clip(median, lower, upper), color='#202020', lw=1.0, ls=(0, (3, 2)))
    ax.text(0.97, 0.94, f"Median = {median:{spec['median_fmt']}}", transform=ax.transAxes,
            ha='right', va='top', fontsize=7.2, color='#303030')
    ax.set_xlabel(spec['xlabel'], labelpad=4)
    ax.set_xlim(lower, upper)
    ax.set_ylim(0, max(counts) * 1.16)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.grid(axis='y', color='#D8D8D8', lw=0.45, alpha=0.72, zorder=0)
    ax.set_axisbelow(True)
    if index == 0:
        ax.set_ylabel('Relative frequency (%)')
    else:
        ax.set_ylabel('')
    if spec.get('log_x', False):
        ax.set_xscale('log')
        ax.set_xticks([1, 3, 6, 12, 24, 72])
        ax.set_xticklabels(['1', '3', '6', '12', '24', '72'])
        ax.minorticks_off()

fig_dist.savefig(DISTRIBUTIONS_PATH, dpi=600, facecolor='white')
plt.show()

fig_space, ax_space = plt.subplots(figsize=(7.1, 5.0), facecolor='white')
fig_space.subplots_adjust(left=0.115, right=0.885, bottom=0.135, top=0.970)
hb = ax_space.hexbin(
    scores[:, 0], scores[:, 1], gridsize=68, mincnt=1,
    cmap=SPACE_CMAP, norm=LogNorm(), linewidths=0, rasterized=True,
)
ax_space.axhline(0, color='#9A9A9A', lw=0.55, zorder=1)
ax_space.axvline(0, color='#9A9A9A', lw=0.55, zorder=1)

arrow_scale = 2.35
for label, color, vector in zip(SHORT_LABELS, COLORS, loadings):
    end_x, end_y = vector[0] * arrow_scale, vector[1] * arrow_scale
    ax_space.annotate('', xy=(end_x, end_y), xytext=(0, 0),
                      arrowprops=dict(arrowstyle='-|>', color=color, lw=1.25, mutation_scale=9), zorder=5)
    horizontal = 'left' if end_x >= 0 else 'right'
    vertical = 'bottom' if end_y >= 0 else 'top'
    ax_space.text(end_x * 1.06, end_y * 1.06, label, color=color, fontsize=7.6,
                  ha=horizontal, va=vertical, fontweight='bold', zorder=6)

x_low, x_high = np.quantile(scores[:, 0], [0.001, 0.999])
y_low, y_high = np.quantile(scores[:, 1], [0.001, 0.999])
x_pad = 0.06 * (x_high - x_low)
y_pad = 0.08 * (y_high - y_low)
ax_space.set_xlim(x_low - x_pad, x_high + x_pad)
ax_space.set_ylim(y_low - y_pad, y_high + y_pad)
ax_space.set_xlabel('PC1')
ax_space.set_ylabel('PC2')
space_cbar = fig_space.colorbar(
    hb, ax=ax_space, fraction=0.030, pad=0.020, shrink=0.82, aspect=28,
)
space_cbar.set_label('Grid cells per hexagon', labelpad=4)
space_cbar.outline.set_linewidth(0.55)
space_cbar.ax.tick_params(labelsize=7.2, length=2.4, width=0.55)
fig_space.savefig(RESPONSE_SPACE_PATH, dpi=600, facecolor='white')
plt.show()
print('Saved:', DISTRIBUTIONS_PATH.relative_to(ROOT))
print('Saved:', RESPONSE_SPACE_PATH.relative_to(ROOT))
